In [ ]:
from collections import Counter, defaultdict
from dataclasses import dataclass
import functools
from io import StringIO
import itertools
from pathlib import Path
import re

import numpy as np
import pandas as pd
import plotly.express as plt
from tqdm import tqdm


DATA_DIR = Path('./data')
DATA_DIR.mkdir(exist_ok=True, parents=True)

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(exist_ok=True)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(exist_ok=True)


In [ ]:

def load_record_to_df(rec_fp: Path) -> pd.DataFrame:
    rec_text_raw = rec_fp.read_text()
    rec_col_names = [
        raw_line.split('|')[-1]
        for raw_line in rec_text_raw[:rec_text_raw.find('Data Table:')].splitlines()
        if re.match(r'^\d', raw_line)
    ]
    rec_raw_text = re.sub(r' +', ',', '\n'.join([raw_line.strip() for raw_line in rec_text_raw.split('sample\n')[1].splitlines()]))
    # print(rec_col_names)
    # print(rec_raw_text[:1000])

    rec_raw_pd_data = f'idx,{",".join(rec_col_names)}\n' + rec_raw_text
    raw_rec_pd = pd.read_csv(StringIO(rec_raw_pd_data), skiprows=0, header=0)
    rec_pd = raw_rec_pd.drop(columns=['idx'])
    return rec_pd


spreadsheet_path_list = list(RAW_DIR.rglob('*.txt'))
print(len(spreadsheet_path_list), 'record files found')

# sp_path = sorted(spreadsheet_path_list, key=lambda x: str(x))[10]
# print(f'Processing {sp_path}')

# filtered_path_list = [fp for fp in sorted(spreadsheet_path_list) if 'hwver' in fp.name]
filtered_path_list = [fp for fp in sorted(spreadsheet_path_list)]
print(f'{len(filtered_path_list)} records after filtering')

rec_dfs = { record_fp.name : load_record_to_df(record_fp) for record_fp in filtered_path_list }
recording = next(iter(rec_dfs.items()))
recording[0], recording[1]


In [ ]:
Counter([(val, len(list(grp))) for val, grp in itertools.groupby(recording[1]['spi_clk'].to_list())])


In [ ]:

merged_df = pd.concat(list(rec_dfs.values())[:])
merged_df


In [ ]:
from scipy.signal import medfilt

# Analyze spikes in channels

def count_sequences(seq: list[int]):
    res = []

    for val, grp in itertools.groupby(seq):
        grp_len = len(list(grp))
        if grp_len < 50:
            res.append((val, grp_len))
    return Counter(res)

# count_sequences(
#     np.asarray()
#     merged_df['spi_clk'].to_list() + 
#     merged_df['spi_cs_n'].to_list() +
#     merged_df['spi_mosi'].to_list() +
#     merged_df['spi_miso'].to_list()
# )

count_sequences(merged_df['spi_clk'].rolling(window=5, center=True, min_periods=1).median().astype(np.int64).to_list())
# count_sequences(medfilt((merged_df['spi_clk'] + merged_df['spi_cs_n']).to_numpy(), kernel_size=5))


In [ ]:
from vcd import VCDWriter


def convert_many_dfs_to_vcd(records: list[pd.DataFrame], target_file: Path):
    with target_file.open('w') as fl:
        with VCDWriter(fl, timescale="1 ns", date="today") as writer:
            col_names = [ "spi_clk", "spi_cs_n", "spi_miso", "spi_mosi" ]

            signals = {}
            for col in col_names:
                signals[col] = writer.register_var(
                    scope="merged_top", 
                    name=col, 
                    var_type="wire", 
                    size=1
                )

            last_states = {col: None for col in col_names}
            REC_SPLIT_DELAY = 2000
            global_timestamp = 0

            for record in tqdm(records):
                for timestamp, row in record.rolling(window=5, center=True, min_periods=1).median().iterrows():
                    cur_timestamp = global_timestamp + timestamp
                    for col in record.columns:
                        current_val = int(row[col])
                        
                        if current_val != last_states[col]:
                            writer.change(signals[col], cur_timestamp, current_val)
                            last_states[col] = current_val
                
                global_timestamp = cur_timestamp
                for col in record.columns:
                    writer.change(signals[col], global_timestamp + 10, 'x')
                for col in record.columns:
                    writer.change(signals[col], global_timestamp + REC_SPLIT_DELAY, 1)
                global_timestamp += REC_SPLIT_DELAY


convert_many_dfs_to_vcd(list(map(lambda x: x[1], rec_dfs.items()))[:], (PROCESSED_DIR / 'result_recording.vcd'))


In [ ]:
break

In [ ]:

# Convert dataframe into VCD file

from vcd import VCDWriter


def convert_df_to_vcd(record: pd.DataFrame, target_file: Path):
    with target_file.open('w') as fl:
        with VCDWriter(fl, timescale="1 ns", date="today") as writer:
            
            signals = {}
            for col in record.columns:
                signals[col] = writer.register_var(
                    scope="top", 
                    name=col, 
                    var_type="wire", 
                    size=1
                )

            last_states = {col: None for col in record.columns}
            
            for timestamp, row in record.iterrows():
                for col in record.columns:
                    current_val = int(row[col])
                    
                    if current_val != last_states[col]:
                        writer.change(signals[col], timestamp, current_val)
                        last_states[col] = current_val

rec_pd = recording[1]
convert_df_to_vcd(rec_pd, (RAW_DIR / 'mcu-w5500-hwver-capture12.txt').with_suffix('.vcd'))


In [ ]:
# record = list(map(lambda x: twos_complement(x[:-1], 16), (rec_pd[ 2**13 : ].to_list())))
cur_record = rec_pd.iloc[ 2**14 : , :].iloc[:4000, 2]

print(cur_record)
plt.line(cur_record)


In [ ]:
def shrink_channel_with_major_vote():
    pass


In [ ]:
chan_list = rec_pd['spi_mosi'].to_list()
